# Workshop: Multiple Tools SRE Agent

## Overview

In this workshop module, you'll expand your Strands Agent with three Kubernetes tools that can investigate complex infrastructure issues. This demonstrates how AI agents can orchestrate multiple tools for comprehensive incident analysis.

### Learning Objectives

By the end of this module, you will:
- Build an agent with multiple specialized tools
- Create a robust FastAPI backend with realistic Kubernetes data
- Implement tool orchestration for complex investigations
- Analyze multi-pod resource usage and correlation

### Prerequisites

- AWS Account with Amazon Bedrock access
- Claude 3 Haiku model enabled in your AWS account
- Python 3.9+ environment
- Completion of Module 0 (Single Tool Agent)

### Architecture

```
┌─────────────────┐    ┌─────────────────────────┐    ┌─────────────────┐
│                 │    │ @tool Functions:         │    │                 │
│ Strands Agent   │───▶│ • get_pod_status        │───▶│ FastAPI Backend │
│                 │    │ • get_pod_events        │    │                 │
│ • Claude Haiku  │    │ • get_pod_resources     │    │ • Pod Data      │
│ • Investigation │    │                         │    │ • Events        │
│ • Orchestration │    │                         │    │ • Resources     │
└─────────────────┘    └─────────────────────────┘    └─────────────────┘
```

**Estimated completion time:** 30 minutes

## Step 1: Environment Setup

Install the required packages for this workshop module.

In [ ]:
%%bash
pip install fastapi uvicorn strands requests --quiet
echo "✅ Packages installed successfully"

In [ ]:
# Import required libraries
from fastapi import FastAPI, HTTPException, Query
from strands import Agent, tool
from strands.models import BedrockModel
from typing import List, Dict, Any, Optional
import uvicorn
import threading
import time
import requests
from datetime import datetime, timedelta
import json

print("✅ Libraries imported successfully")

## Step 2: Create Enhanced Infrastructure Backend

Create a FastAPI backend with multiple endpoints that simulates a Kubernetes cluster with realistic pod data, events, and resource metrics.

In [ ]:
# Create FastAPI application with realistic Kubernetes data
app = FastAPI(title="Kubernetes API Simulator", version="1.0.0")

# Realistic pod data with expanded multi-pod scenario
PODS_DATA = {
    "pods": [
        {
            "name": "payment-service-7d4f8-x5m1q",
            "namespace": "production",
            "status": "CrashLoopBackOff",
            "ready": False,
            "restart_count": 15,
            "cpu_usage": "25%",
            "memory_usage": "98%",
            "node": "worker-node-2",
            "last_restart": "2024-01-15T14:24:30Z",
            "containers": [
                {
                    "name": "payment-api",
                    "image": "payment-service:v1.2.3",
                    "status": "Waiting",
                    "reason": "CrashLoopBackOff",
                    "message": "Back-off 5m0s restarting failed container"
                }
            ]
        },
        {
            "name": "payment-service-7d4f8-j9k7l",
            "namespace": "production",
            "status": "Running",
            "ready": True,
            "restart_count": 2,
            "cpu_usage": "78%",
            "memory_usage": "87%",
            "node": "worker-node-1",
            "last_restart": "2024-01-15T12:15:45Z",
            "containers": [
                {
                    "name": "payment-api",
                    "image": "payment-service:v1.2.3",
                    "status": "Running",
                    "reason": "Started",
                    "message": "Container running but approaching memory limits"
                }
            ]
        },
        {
            "name": "user-service-9k2x1-y6n2r",
            "namespace": "production",
            "status": "Running",
            "ready": True,
            "restart_count": 0,
            "cpu_usage": "32%",
            "memory_usage": "64%",
            "node": "worker-node-1",
            "last_restart": None,
            "containers": [
                {
                    "name": "user-api",
                    "image": "user-service:v1.1.0",
                    "status": "Running",
                    "reason": "Started",
                    "message": "Container started successfully"
                }
            ]
        },
        {
            "name": "database-service-3r5t6-h8j9k",
            "namespace": "production",
            "status": "Running",
            "ready": True,
            "restart_count": 0,
            "cpu_usage": "45%",
            "memory_usage": "72%",
            "node": "worker-node-2",
            "last_restart": None,
            "containers": [
                {
                    "name": "postgres",
                    "image": "postgres:14.5",
                    "status": "Running",
                    "reason": "Started",
                    "message": "Container started successfully"
                }
            ]
        }
    ]
}

# Detailed pod events data
EVENTS_DATA = {
    "events": {
        "payment-service-7d4f8-x5m1q": [
            {
                "type": "Warning",
                "reason": "OutOfMemoryKilled",
                "message": "Container payment-api was killed due to OOM (Out of Memory). Memory cgroup usage exceeds configured limit.",
                "count": 15,
                "timestamp": "2024-01-15T14:24:28Z"
            },
            {
                "type": "Warning",
                "reason": "BackOff",
                "message": "Back-off restarting failed container payment-api in pod payment-service-7d4f8-x5m1q",
                "count": 12,
                "timestamp": "2024-01-15T14:24:45Z"
            },
            {
                "type": "Normal",
                "reason": "Pulled",
                "message": "Successfully pulled image 'payment-service:v1.2.3'",
                "count": 16,
                "timestamp": "2024-01-15T14:19:20Z"
            },
            {
                "type": "Warning",
                "reason": "FailedMemoryAllocation",
                "message": "Memory allocation failed with error: Cannot allocate memory for heap",
                "count": 8,
                "timestamp": "2024-01-15T14:22:15Z"
            }
        ],
        "payment-service-7d4f8-j9k7l": [
            {
                "type": "Warning",
                "reason": "MemoryPressure",
                "message": "Container payment-api is approaching memory limit. Current usage: 87% of allocated memory.",
                "count": 5,
                "timestamp": "2024-01-15T14:20:05Z"
            },
            {
                "type": "Warning",
                "reason": "HighCPUUsage",
                "message": "Container payment-api is using high CPU: 78% of allocated CPU.",
                "count": 8,
                "timestamp": "2024-01-15T14:15:30Z"
            },
            {
                "type": "Normal",
                "reason": "Started",
                "message": "Started container payment-api",
                "count": 3,
                "timestamp": "2024-01-15T12:15:50Z"
            }
        ],
        "user-service-9k2x1-y6n2r": [
            {
                "type": "Normal",
                "reason": "Pulled",
                "message": "Successfully pulled image 'user-service:v1.1.0'",
                "count": 1,
                "timestamp": "2024-01-15T08:30:15Z"
            },
            {
                "type": "Normal",
                "reason": "Created",
                "message": "Created container user-api",
                "count": 1,
                "timestamp": "2024-01-15T08:30:18Z"
            },
            {
                "type": "Normal",
                "reason": "Started",
                "message": "Started container user-api",
                "count": 1,
                "timestamp": "2024-01-15T08:30:20Z"
            }
        ],
        "database-service-3r5t6-h8j9k": [
            {
                "type": "Normal",
                "reason": "Pulled",
                "message": "Successfully pulled image 'postgres:14.5'",
                "count": 1,
                "timestamp": "2024-01-14T23:15:10Z"
            },
            {
                "type": "Normal",
                "reason": "Created",
                "message": "Created container postgres",
                "count": 1,
                "timestamp": "2024-01-14T23:15:12Z"
            },
            {
                "type": "Normal",
                "reason": "Started",
                "message": "Started container postgres",
                "count": 1,
                "timestamp": "2024-01-14T23:15:15Z"
            }
        ]
    }
}

# Detailed resource metrics data (historical)
RESOURCES_DATA = {
    "resources": {
        "payment-service-7d4f8-x5m1q": {
            "memory": [
                {"timestamp": "2024-01-15T12:00:00Z", "value": "65%"},
                {"timestamp": "2024-01-15T12:30:00Z", "value": "72%"},
                {"timestamp": "2024-01-15T13:00:00Z", "value": "79%"},
                {"timestamp": "2024-01-15T13:30:00Z", "value": "85%"},
                {"timestamp": "2024-01-15T14:00:00Z", "value": "91%"},
                {"timestamp": "2024-01-15T14:20:00Z", "value": "98%"},
                {"timestamp": "2024-01-15T14:24:30Z", "value": "100%"}
            ],
            "cpu": [
                {"timestamp": "2024-01-15T12:00:00Z", "value": "18%"},
                {"timestamp": "2024-01-15T12:30:00Z", "value": "20%"},
                {"timestamp": "2024-01-15T13:00:00Z", "value": "22%"},
                {"timestamp": "2024-01-15T13:30:00Z", "value": "24%"},
                {"timestamp": "2024-01-15T14:00:00Z", "value": "25%"},
                {"timestamp": "2024-01-15T14:24:30Z", "value": "25%"}
            ],
            "limits": {"memory": "512Mi", "cpu": "500m"},
            "requests": {"memory": "256Mi", "cpu": "250m"}
        },
        "payment-service-7d4f8-j9k7l": {
            "memory": [
                {"timestamp": "2024-01-15T12:00:00Z", "value": "55%"},
                {"timestamp": "2024-01-15T12:30:00Z", "value": "61%"},
                {"timestamp": "2024-01-15T13:00:00Z", "value": "68%"},
                {"timestamp": "2024-01-15T13:30:00Z", "value": "73%"},
                {"timestamp": "2024-01-15T14:00:00Z", "value": "79%"},
                {"timestamp": "2024-01-15T14:20:00Z", "value": "87%"}
            ],
            "cpu": [
                {"timestamp": "2024-01-15T12:00:00Z", "value": "45%"},
                {"timestamp": "2024-01-15T12:30:00Z", "value": "52%"},
                {"timestamp": "2024-01-15T13:00:00Z", "value": "58%"},
                {"timestamp": "2024-01-15T13:30:00Z", "value": "65%"},
                {"timestamp": "2024-01-15T14:00:00Z", "value": "72%"},
                {"timestamp": "2024-01-15T14:20:00Z", "value": "78%"}
            ],
            "limits": {"memory": "512Mi", "cpu": "500m"},
            "requests": {"memory": "256Mi", "cpu": "250m"}
        },
        "user-service-9k2x1-y6n2r": {
            "memory": [
                {"timestamp": "2024-01-15T12:00:00Z", "value": "58%"},
                {"timestamp": "2024-01-15T12:30:00Z", "value": "60%"},
                {"timestamp": "2024-01-15T13:00:00Z", "value": "62%"},
                {"timestamp": "2024-01-15T13:30:00Z", "value": "63%"},
                {"timestamp": "2024-01-15T14:00:00Z", "value": "63%"},
                {"timestamp": "2024-01-15T14:20:00Z", "value": "64%"}
            ],
            "cpu": [
                {"timestamp": "2024-01-15T12:00:00Z", "value": "30%"},
                {"timestamp": "2024-01-15T12:30:00Z", "value": "31%"},
                {"timestamp": "2024-01-15T13:00:00Z", "value": "31%"},
                {"timestamp": "2024-01-15T13:30:00Z", "value": "32%"},
                {"timestamp": "2024-01-15T14:00:00Z", "value": "32%"},
                {"timestamp": "2024-01-15T14:20:00Z", "value": "32%"}
            ],
            "limits": {"memory": "256Mi", "cpu": "250m"},
            "requests": {"memory": "128Mi", "cpu": "100m"}
        },
        "database-service-3r5t6-h8j9k": {
            "memory": [
                {"timestamp": "2024-01-15T12:00:00Z", "value": "70%"},
                {"timestamp": "2024-01-15T12:30:00Z", "value": "70%"},
                {"timestamp": "2024-01-15T13:00:00Z", "value": "71%"},
                {"timestamp": "2024-01-15T13:30:00Z", "value": "71%"},
                {"timestamp": "2024-01-15T14:00:00Z", "value": "72%"},
                {"timestamp": "2024-01-15T14:20:00Z", "value": "72%"}
            ],
            "cpu": [
                {"timestamp": "2024-01-15T12:00:00Z", "value": "40%"},
                {"timestamp": "2024-01-15T12:30:00Z", "value": "42%"},
                {"timestamp": "2024-01-15T13:00:00Z", "value": "44%"},
                {"timestamp": "2024-01-15T13:30:00Z", "value": "44%"},
                {"timestamp": "2024-01-15T14:00:00Z", "value": "45%"},
                {"timestamp": "2024-01-15T14:20:00Z", "value": "45%"}
            ],
            "limits": {"memory": "1Gi", "cpu": "1000m"},
            "requests": {"memory": "512Mi", "cpu": "500m"}
        }
    }
}

@app.get("/health")
def health_check():
    """Health check endpoint"""
    return {"status": "healthy", "service": "kubernetes-api"}

@app.get("/pods")
def get_pods():
    """Get all pods in the cluster"""
    return PODS_DATA

@app.get("/pods/{pod_name}/events")
def get_pod_events(pod_name: str):
    """Get detailed events for a specific pod"""
    if pod_name not in EVENTS_DATA["events"]:
        raise HTTPException(status_code=404, detail=f"Pod {pod_name} not found")
    return {"events": EVENTS_DATA["events"][pod_name]}

@app.get("/pods/{pod_name}/resources")
def get_pod_resource_metrics(pod_name: str):
    """Get detailed resource metrics for a specific pod"""
    if pod_name not in RESOURCES_DATA["resources"]:
        raise HTTPException(status_code=404, detail=f"Pod {pod_name} not found")
    return {"resources": RESOURCES_DATA["resources"][pod_name]}

print("✅ Enhanced FastAPI backend created with pods, events, and resource metrics")

In [ ]:
# Start the FastAPI server in background
def start_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="error")

server_thread = threading.Thread(target=start_server, daemon=True)
server_thread.start()

# Wait for server startup
time.sleep(3)

# Verify server is running
try:
    response = requests.get("http://127.0.0.1:8000/health", timeout=5)
    if response.status_code == 200:
        print("✅ Backend server running at http://127.0.0.1:8000")
        print(f"   Health status: {response.json()['status']}")
    else:
        print(f"❌ Server health check failed: {response.status_code}")
except Exception as e:
    print(f"❌ Cannot connect to server: {e}")

## Step 3: Create Multiple Strands Agent Tools

Define three tools that the Strands Agent can use to investigate pod issues, events, and resource usage.

In [ ]:
@tool
def get_pod_status(namespace: str = "production") -> str:
    """
    Get detailed status information for Kubernetes pods in the specified namespace.
    
    Args:
        namespace: Kubernetes namespace to query (default: production)
        
    Returns:
        Comprehensive pod status including health, resource usage, and recent events
    """
    try:
        response = requests.get("http://127.0.0.1:8000/pods", timeout=10)
        response.raise_for_status()
        data = response.json()
        
        # Filter pods by namespace
        filtered_pods = [pod for pod in data["pods"] if pod["namespace"] == namespace]
        
        if not filtered_pods:
            return f"No pods found in namespace '{namespace}'"
        
        result = f"Found {len(filtered_pods)} pods in '{namespace}' namespace:\n\n"
        
        for pod in filtered_pods:
            status_icon = "❌" if not pod["ready"] else "✅"
            result += f"{status_icon} Pod: {pod['name']}\n"
            result += f"   Status: {pod['status']} (Ready: {pod['ready']})\n"
            result += f"   Restarts: {pod['restart_count']}\n"
            result += f"   Resource Usage: CPU {pod['cpu_usage']}, Memory {pod['memory_usage']}\n"
            result += f"   Node: {pod['node']}\n"
            
            if pod.get('last_restart'):
                result += f"   Last Restart: {pod['last_restart']}\n"
            
            # Include container details
            if pod.get('containers'):
                result += f"   Containers:\n"
                for container in pod['containers']:
                    result += f"     - {container['name']}: {container['status']} ({container['reason']})\n"
            
            result += "\n"
            
        return result
        
    except requests.RequestException as e:
        return f"Error querying Kubernetes API: {e}"
    except Exception as e:
        return f"Unexpected error: {e}"

@tool
def get_pod_events(pod_name: str) -> str:
    """
    Get detailed events and warnings for a specific Kubernetes pod.
    
    Args:
        pod_name: Name of the Kubernetes pod to query
        
    Returns:
        Chronological list of events with timestamps, types, and messages
    """
    try:
        # First check if pod exists
        pod_response = requests.get("http://127.0.0.1:8000/pods", timeout=10)
        pod_response.raise_for_status()
        pod_data = pod_response.json()
        
        pod_exists = False
        for pod in pod_data["pods"]:
            if pod["name"] == pod_name:
                pod_exists = True
                break
                
        if not pod_exists:
            return f"Pod '{pod_name}' not found in the cluster"
        
        # Get events for the pod
        events_response = requests.get(f"http://127.0.0.1:8000/pods/{pod_name}/events", timeout=10)
        events_response.raise_for_status()
        events_data = events_response.json()
        
        if not events_data["events"]:
            return f"No events found for pod '{pod_name}'"
        
        # Sort events by timestamp (newest first)
        sorted_events = sorted(events_data["events"], key=lambda x: x["timestamp"], reverse=True)
        
        result = f"Events for pod '{pod_name}' (newest first):\n\n"
        
        for event in sorted_events:
            # Format timestamp nicely
            event_time = event["timestamp"]
            event_type = event["type"]
            event_icon = "⚠️" if event_type == "Warning" else "ℹ️" if event_type == "Normal" else "🔴"
            
            result += f"{event_icon} [{event_time}] {event_type}: {event['reason']}\n"
            result += f"    {event['message']}\n"
            result += f"    Occurred {event['count']} times\n\n"
            
        return result
        
    except requests.RequestException as e:
        return f"Error querying Kubernetes API: {e}"
    except Exception as e:
        return f"Unexpected error: {e}"

@tool
def get_pod_resources(pod_name: str) -> str:
    """
    Get detailed resource metrics and history for a specific Kubernetes pod.
    
    Args:
        pod_name: Name of the Kubernetes pod to query
        
    Returns:
        Historical CPU and memory usage, limits, and requests
    """
    try:
        # First check if pod exists
        pod_response = requests.get("http://127.0.0.1:8000/pods", timeout=10)
        pod_response.raise_for_status()
        pod_data = pod_response.json()
        
        pod_exists = False
        for pod in pod_data["pods"]:
            if pod["name"] == pod_name:
                pod_exists = True
                break
                
        if not pod_exists:
            return f"Pod '{pod_name}' not found in the cluster"
        
        # Get resource metrics for the pod
        metrics_response = requests.get(f"http://127.0.0.1:8000/pods/{pod_name}/resources", timeout=10)
        metrics_response.raise_for_status()
        metrics_data = metrics_response.json()
        
        resources = metrics_data["resources"]
        
        result = f"Resource metrics for pod '{pod_name}':\n\n"
        
        # Add resource limits and requests
        result += "Resource Configuration:\n"
        result += f"  Memory Limit:   {resources['limits']['memory']}\n"
        result += f"  Memory Request: {resources['requests']['memory']}\n"
        result += f"  CPU Limit:      {resources['limits']['cpu']}\n"
        result += f"  CPU Request:    {resources['requests']['cpu']}\n\n"
        
        # Memory usage over time
        result += "Memory Usage History (newest first):\n"
        for entry in sorted(resources["memory"], key=lambda x: x["timestamp"], reverse=True):
            timestamp = entry["timestamp"]
            value = entry["value"]
            result += f"  {timestamp}: {value}\n"
        
        result += "\nCPU Usage History (newest first):\n"
        for entry in sorted(resources["cpu"], key=lambda x: x["timestamp"], reverse=True):
            timestamp = entry["timestamp"]
            value = entry["value"]
            result += f"  {timestamp}: {value}\n"
            
        # Add trend analysis
        memory_values = [int(entry["value"].rstrip("%")) for entry in resources["memory"]]
        cpu_values = [int(entry["value"].rstrip("%")) for entry in resources["cpu"]]
        
        memory_trend = "increasing" if memory_values[-1] > memory_values[0] else "decreasing" if memory_values[-1] < memory_values[0] else "stable"
        cpu_trend = "increasing" if cpu_values[-1] > cpu_values[0] else "decreasing" if cpu_values[-1] < cpu_values[0] else "stable"
        
        result += "\nTrend Analysis:\n"
        result += f"  Memory usage is {memory_trend}\n"
        result += f"  CPU usage is {cpu_trend}\n"
        
        return result
        
    except requests.RequestException as e:
        return f"Error querying Kubernetes API: {e}"
    except Exception as e:
        return f"Unexpected error: {e}"

print("✅ Three Strands tool functions created:")
print("   1. get_pod_status() - Pod health and status overview")
print("   2. get_pod_events() - Detailed event history")
print("   3. get_pod_resources() - Resource usage metrics and trends")

## Step 4: Initialize Strands Agent with Multiple Tools

Create a Strands Agent with access to all three tools for complex investigations.

In [ ]:
# Initialize Bedrock model and Strands Agent
try:
    # Create Bedrock model instance
    model = BedrockModel(model_id="us.anthropic.claude-3-haiku-20240307-v1:0")
    
    # Create Strands Agent with professional SRE system prompt
    agent = Agent(
        model=model,
        tools=[get_pod_status, get_pod_events, get_pod_resources],
        system_prompt="""You are an expert Site Reliability Engineer (SRE) investigating infrastructure issues.

        Your responsibilities:
        1. Use available tools to systematically gather infrastructure data
        2. Analyze the information to identify root causes and contributing factors
        3. Provide specific, actionable recommendations for resolution
        4. Explain your reasoning clearly and concisely
        5. Focus on immediate fixes and preventive measures

        When orchestrating multiple tools:
        - Begin with broad status checks to identify affected services
        - Examine event logs for specific error patterns
        - Analyze resource metrics to identify usage patterns and trends
        - Correlate information across tools to build a complete picture
        - Consider relationships between different services

        Be direct, technical, and solution-focused in your analysis."""
    )
    
    print("✅ Strands Agent initialized successfully")
    print(f"   Model: Claude 3 Haiku (us.anthropic.claude-3-haiku-20240307-v1:0)")
    print(f"   Tools: 3 tools available (get_pod_status, get_pod_events, get_pod_resources)")
    print(f"   Framework: Strands Agents")
    
except Exception as e:
    print(f"❌ Failed to initialize Strands Agent: {e}")
    print("\nTroubleshooting steps:")
    print("1. Verify AWS credentials: aws configure list")
    print("2. Check Bedrock access in your AWS region")
    print("3. Ensure Claude 3 Haiku model is enabled")
    agent = None

## Step 5: Run Complex Investigation

Let the Strands Agent investigate a complex multi-pod issue by orchestrating multiple tools.

In [ ]:
if agent:
    print("🚨 PRODUCTION INCIDENT")
    print("=" * 40)
    print("ALERT: Payment service degradation detected!")
    print("Impact: Users reporting slow payment processing and some failures")
    print("Priority: P1 - Critical")
    print("\nStarting AI investigation...\n")
    
    # Record investigation start time
    start_time = time.time()
    
    # Run the investigation
    incident_description = (
        "URGENT: We're seeing degraded performance in our payment service. "
        "Users are reporting that payments are taking a long time to process "
        "and some are failing completely. This started approximately 30 minutes ago "
        "and is getting worse. Please investigate all services in the production "
        "namespace and determine what's causing this issue."
    )
    
    try:
        response = agent(incident_description)
        investigation_time = round(time.time() - start_time, 1)
        
        print(f"⚡ Investigation completed in {investigation_time} seconds")
        print("\n" + "=" * 60)
        print("AI INVESTIGATION RESULTS")
        print("=" * 60)
        
        # Access the response correctly
        if hasattr(response, 'content'):
            print(response.content)
        elif hasattr(response, 'message'):
            print(response.message)
        else:
            print(str(response))
        
        print("\n" + "=" * 60)
        
    except Exception as e:
        print(f"❌ Investigation failed: {e}")
        
else:
    print("❌ Cannot run investigation - Strands Agent not initialized")
    print("Please check the previous steps for errors")

## Step 6: Review Results

Analyze what the Strands Agent accomplished with multiple tools.

In [ ]:
if agent and 'response' in locals():
    print("📊 INVESTIGATION ANALYSIS")
    print("=" * 30)
    
    # Show what the agent accomplished
    print("\nWhat the AI Agent did:")
    print("✅ Orchestrated multiple tools autonomously")
    print("✅ Gathered comprehensive pod status information")
    print("✅ Examined detailed event logs for error patterns")
    print("✅ Analyzed resource usage trends over time")
    print("✅ Correlated information across tools")
    print("✅ Identified both symptoms and root causes")
    print("✅ Provided prioritized action items")
    
    # Compare to manual investigation
    print("\nComparison to Manual Investigation:")
    print(f"⚡ AI Investigation: {investigation_time} seconds")
    print("⏳ Manual Investigation: 30-45 minutes typical")
    print("📈 Speed Improvement: ~98% faster")
    print("🔍 Cross-Service Correlation: Automated vs. manual correlation")
    
    print("\nKey Benefits Demonstrated:")
    print("• Complex tool orchestration")
    print("• Multi-pod analysis capabilities")
    print("• Historical trend analysis")
    print("• Inter-service correlation")
    print("• Root cause identification")
    
else:
    print("Investigation results not available")
    print("Please ensure the previous steps completed successfully")

## Summary and Next Steps

### What You Accomplished

In this workshop module, you:

1. **Created an enhanced FastAPI backend** with pods, events, and resource metrics
2. **Built a Strands Agent** with multiple specialized tools
3. **Implemented tool orchestration** for complex multi-pod analysis
4. **Demonstrated advanced incident response** capabilities

### Key Learnings

- **Multiple @tool functions** enable comprehensive investigations
- **Tool orchestration** allows the agent to gather and correlate information
- **Resource trend analysis** helps identify developing problems
- **Cross-service correlation** provides deeper insights

### Workshop Progression

This module demonstrated multi-tool orchestration. The complete workshop series continues with:

- **Module 2**: Secure gateway integration with MCP protocol  
- **Module 3**: Multi-agent architecture with specialist agents
- **Module 4**: Memory integration for persistent learning
- **Module 5**: Production deployment to AgentCore Runtime

### Resources

- [Strands Documentation](https://strands.dev)
- [Amazon Bedrock User Guide](https://docs.aws.amazon.com/bedrock/)
- [FastAPI Documentation](https://fastapi.tiangolo.com/)

---

**Next**: Continue to Module 2 to implement secure gateway integration with MCP protocol.